# Packages


In [3]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import networkx as nx
import leidenalg as la
import igraph as ig
import sys
import glob
from tqdm import tqdm
from dotenv import load_dotenv
import os
# Load environment variables from .env file
load_dotenv()

# Access environment variables
project_root = os.getenv('PYTHONPATH')
output_dir = os.path.join(project_root, os.getenv('OUTPUT_DIR'))
data_dir = os.path.join(project_root, os.getenv('DATA_DIR'))
src_dir = os.path.join(project_root, os.getenv('SRC_DIR'))



from src.network.creation.PartitionCreator import PartitionCreator
from src.network.analysis.NetworkAnalyzer import CommunityExplorer, FullExplorer
from src.network.analysis.NetworkAnalyzerUtils import NetworkAnalyzerUtils

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# load df and graphs


In [5]:
# load all files in a dictionary with params as keys and graph as values
graph_dir = data_dir + "/05-graphs/weighted-knn-citation-graph"
embeddings_dir = data_dir + "/04-embeddings"
params_graph_dict, df = NetworkAnalyzerUtils().load_graph_files(
    graph_dir, embeddings_dir
)

# remove the date part of the keys
params_graph_dict = {
    k.replace("_knn_citation_20250401.graphml", ""): v
    for k, v in params_graph_dict.items()
}

print(params_graph_dict.keys())

dict_keys(['alpha0.3_k10', 'alpha0.3_k15', 'alpha0.3_k20', 'alpha0.3_k5', 'alpha0.5_k10', 'alpha0.5_k15', 'alpha0.5_k20', 'alpha0.5_k5'])


In [7]:
df["title_abstract"] = df.apply(
    lambda x: (
        x["title"] + ". " + str(x["abstract"])
        if pd.notnull(x["abstract"])
        else x["title"]
    ),
    axis=1,
)

In [13]:
iterations = 25
graph_summary_df_dict = {}

last_selection_params = [
    "alpha0.3_k20_res0.006",
    "alpha0.5_k10_res0.002",
    "alpha0.3_k5_res0.001",
    "alpha0.3_k20_res0.005",
    "alpha0.3_k10_res0.002",
]

# Initialize df_clusters with just the eid column from df
df_clusters = df[["eid"]].copy()

for params in last_selection_params:
    G = params_graph_dict[params.split("_res")[0]]
    resolution = pd.to_numeric(params.split("res")[1])
    column_name = f"cluster_{params}"

    pc = PartitionCreator(G, df)
    pc.create_partition_from_cmpvertexpartition(
        n_iterations=iterations,
        resolution_parameter=resolution,
        verbose=False,
        cluster_column_name=column_name,
        centrality_column_name=f"centrality_{params}",
    )

    # Extract just the eid and cluster column
    df_subset = pc.dataframe[["eid", column_name]]

    # Merge with df_clusters on eid
    df_clusters = df_clusters.merge(df_subset, on="eid", how="left")

In [14]:
df_clusters

,eid,cluster_alpha0.3_k20_res0.006,cluster_alpha0.5_k10_res0.002,cluster_alpha0.3_k5_res0.001,cluster_alpha0.3_k20_res0.005,cluster_alpha0.3_k10_res0.002
0,2-s2.0-0020425640,103,23,2,91,20
1,2-s2.0-0019951467,74,74,72,80,69
2,2-s2.0-0020059465,8,15,63,9,10
3,2-s2.0-0019961783,115,79,78,62,42
4,2-s2.0-0019992213,5,86,1,4,2
...,...,...,...,...,...,...
38956,2-s2.0-85214339343,126,99,93,114,111
38957,2-s2.0-85212792674,79,57,43,72,73
38958,2-s2.0-85218337443,92,19,18,11,14
38959,2-s2.0-85218783956,169,49,52,56,43


In [16]:
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score


def jaccard_similarity(set1, set2):
    """Calculate Jaccard similarity between two sets"""
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union > 0 else 0


def create_overlap_matrix(df_clusters, cluster_col1, cluster_col2, min_size=100):
    """
    Create matrix of Jaccard similarities between all cluster pairs

    Parameters:
    df_clusters: DataFrame with eid and cluster assignments
    cluster_col1: name of first cluster column
    cluster_col2: name of second cluster column
    min_size: minimum cluster size to include

    Returns:
    overlap_matrix: matrix of Jaccard similarities
    large_clusters1: list of cluster IDs from solution 1 meeting size threshold
    large_clusters2: list of cluster IDs from solution 2 meeting size threshold
    """
    # Get clusters meeting size threshold
    cluster1_sizes = df_clusters[cluster_col1].value_counts()
    cluster2_sizes = df_clusters[cluster_col2].value_counts()

    large_clusters1 = cluster1_sizes[cluster1_sizes >= min_size].index.tolist()
    large_clusters2 = cluster2_sizes[cluster2_sizes >= min_size].index.tolist()

    # Create overlap matrix
    n1 = len(large_clusters1)
    n2 = len(large_clusters2)
    overlap_matrix = np.zeros((n1, n2))

    for i, c1 in enumerate(large_clusters1):
        papers_c1 = set(df_clusters[df_clusters[cluster_col1] == c1]["eid"])

        for j, c2 in enumerate(large_clusters2):
            papers_c2 = set(df_clusters[df_clusters[cluster_col2] == c2]["eid"])
            overlap_matrix[i, j] = jaccard_similarity(papers_c1, papers_c2)

    return overlap_matrix, large_clusters1, large_clusters2


def match_clusters_and_assess_stability(
    df_clusters, cluster_col1, cluster_col2, min_size=100
):
    """
    Match clusters between two solutions and assess stability

    Uses Hungarian algorithm to find optimal matching that maximizes total overlap
    """

    # Create overlap matrix
    overlap_matrix, clusters1, clusters2 = create_overlap_matrix(
        df_clusters, cluster_col1, cluster_col2, min_size
    )

    print(f"\n{'='*70}")
    print(f"COMPARING: {cluster_col1}")
    print(f"      VS:  {cluster_col2}")
    print(f"{'='*70}")
    print(f"Solution 1: {len(clusters1)} clusters with ≥{min_size} publications")
    print(f"Solution 2: {len(clusters2)} clusters with ≥{min_size} publications")

    if len(clusters1) == 0 or len(clusters2) == 0:
        print("WARNING: One or both solutions have no large clusters!")
        return None, None

    # Use Hungarian algorithm to find optimal matching
    # We want to MAXIMIZE overlap, so negate the matrix (algorithm minimizes)
    row_ind, col_ind = linear_sum_assignment(-overlap_matrix)

    # Collect results for matched pairs
    matches = []
    for i, j in zip(row_ind, col_ind):
        c1 = clusters1[i]
        c2 = clusters2[j]
        jaccard = overlap_matrix[i, j]

        size1 = (df_clusters[cluster_col1] == c1).sum()
        size2 = (df_clusters[cluster_col2] == c2).sum()

        matches.append(
            {
                "cluster_solution1": c1,
                "cluster_solution2": c2,
                "size_solution1": size1,
                "size_solution2": size2,
                "jaccard_similarity": jaccard,
                "overlap_pct": jaccard * 100,
            }
        )

    results_df = pd.DataFrame(matches).sort_values(
        "jaccard_similarity", ascending=False
    )

    # Calculate statistics
    mean_jaccard = results_df["jaccard_similarity"].mean()
    median_jaccard = results_df["jaccard_similarity"].median()
    high_overlap = (results_df["jaccard_similarity"] > 0.8).sum()
    moderate_overlap = (results_df["jaccard_similarity"] > 0.6).sum()

    print(f"\n{'='*70}")
    print(f"STABILITY METRICS FOR MATCHED CLUSTERS")
    print(f"{'='*70}")
    print(f"Mean Jaccard similarity:           {mean_jaccard:.3f}")
    print(f"Median Jaccard similarity:         {median_jaccard:.3f}")
    print(
        f"Clusters with >80% overlap:        {high_overlap}/{len(results_df)} ({high_overlap/len(results_df)*100:.1f}%)"
    )
    print(
        f"Clusters with >60% overlap:        {moderate_overlap}/{len(results_df)} ({moderate_overlap/len(results_df)*100:.1f}%)"
    )

    # Global agreement metrics (on full dataset, not just large clusters)
    ari = adjusted_rand_score(df_clusters[cluster_col1], df_clusters[cluster_col2])
    nmi = normalized_mutual_info_score(
        df_clusters[cluster_col1], df_clusters[cluster_col2]
    )

    print(f"\nGLOBAL AGREEMENT METRICS (all publications):")
    print(f"Adjusted Rand Index:               {ari:.3f}")
    print(f"Normalized Mutual Information:     {nmi:.3f}")

    # Show examples
    print(f"\n{'='*70}")
    print("TOP 10 MOST STABLE CLUSTER MATCHES:")
    print(f"{'='*70}")
    display_cols = [
        "cluster_solution1",
        "cluster_solution2",
        "size_solution1",
        "size_solution2",
        "jaccard_similarity",
        "overlap_pct",
    ]
    print(results_df[display_cols].head(10).to_string(index=False))

    if len(results_df) > 10:
        print(f"\n{'='*70}")
        print("TOP 10 LEAST STABLE CLUSTER MATCHES:")
        print(f"{'='*70}")
        print(results_df[display_cols].tail(10).to_string(index=False))

    return results_df, {
        "ARI": ari,
        "NMI": nmi,
        "mean_jaccard": mean_jaccard,
        "median_jaccard": median_jaccard,
    }


def compare_all_solutions(df_clusters, cluster_columns, min_size=100):
    """
    Compare all pairs of clustering solutions

    Parameters:
    df_clusters: DataFrame with eid and multiple cluster columns
    cluster_columns: list of cluster column names to compare
    min_size: minimum cluster size threshold

    Returns:
    summary_df: DataFrame with pairwise comparison metrics
    all_results: dict with detailed results for each comparison
    """
    from itertools import combinations

    summary_results = []
    all_results = {}

    comparisons = list(combinations(cluster_columns, 2))
    print(f"\n{'#'*70}")
    print(f"RUNNING {len(comparisons)} PAIRWISE COMPARISONS")
    print(f"{'#'*70}\n")

    for idx, (col1, col2) in enumerate(comparisons, 1):
        print(f"\n[{idx}/{len(comparisons)}]")

        results_df, metrics = match_clusters_and_assess_stability(
            df_clusters, col1, col2, min_size
        )

        if results_df is not None:
            summary_results.append(
                {
                    "solution1": col1,
                    "solution2": col2,
                    "n_matched_clusters": len(results_df),
                    "mean_jaccard": metrics["mean_jaccard"],
                    "median_jaccard": metrics["median_jaccard"],
                    "pct_high_overlap_80": (
                        results_df["jaccard_similarity"] > 0.8
                    ).sum()
                    / len(results_df)
                    * 100,
                    "pct_moderate_overlap_60": (
                        results_df["jaccard_similarity"] > 0.6
                    ).sum()
                    / len(results_df)
                    * 100,
                    "ARI": metrics["ARI"],
                    "NMI": metrics["NMI"],
                }
            )

            all_results[f"{col1}_vs_{col2}"] = results_df

    summary_df = pd.DataFrame(summary_results)

    # Print overall summary
    print(f"\n{'#'*70}")
    print(f"OVERALL SUMMARY ACROSS ALL COMPARISONS")
    print(f"{'#'*70}")
    print(
        f"\nMean Jaccard similarity across all comparisons: {summary_df['mean_jaccard'].mean():.3f}"
    )
    print(
        f"Range: {summary_df['mean_jaccard'].min():.3f} - {summary_df['mean_jaccard'].max():.3f}"
    )
    print(
        f"\nAverage % of clusters with >80% overlap: {summary_df['pct_high_overlap_80'].mean():.1f}%"
    )
    print(
        f"Average % of clusters with >60% overlap: {summary_df['pct_moderate_overlap_60'].mean():.1f}%"
    )
    print(f"\nMean ARI across all comparisons: {summary_df['ARI'].mean():.3f}")
    print(f"Mean NMI across all comparisons: {summary_df['NMI'].mean():.3f}")

    return summary_df, all_results


# ============================================================================
# USAGE
# ============================================================================

# List your cluster columns
cluster_columns = [
    "cluster_alpha0.3_k20_res0.006",
    "cluster_alpha0.5_k10_res0.002",
    "cluster_alpha0.3_k5_res0.001",
    "cluster_alpha0.3_k20_res0.005",
    "cluster_alpha0.3_k10_res0.002",
]

# Run comprehensive comparison
summary_df, all_results = compare_all_solutions(
    df_clusters, cluster_columns, min_size=100
)

# Save results
summary_df.to_csv("cluster_stability_summary.csv", index=False)

print("\n" + "=" * 70)
print("SUMMARY TABLE:")
print("=" * 70)
print(summary_df.to_string(index=False))

# Example: Look at specific comparison in detail
print("\n" + "=" * 70)
print("DETAILED RESULTS: Your chosen solution vs alternative")
print("=" * 70)


######################################################################
RUNNING 10 PAIRWISE COMPARISONS
######################################################################


[1/10]

COMPARING: cluster_alpha0.3_k20_res0.006
      VS:  cluster_alpha0.5_k10_res0.002
Solution 1: 124 clusters with ≥100 publications
Solution 2: 94 clusters with ≥100 publications

STABILITY METRICS FOR MATCHED CLUSTERS
Mean Jaccard similarity:           0.574
Median Jaccard similarity:         0.571
Clusters with >80% overlap:        15/94 (16.0%)
Clusters with >60% overlap:        42/94 (44.7%)

GLOBAL AGREEMENT METRICS (all publications):
Adjusted Rand Index:               0.545
Normalized Mutual Information:     0.789

TOP 10 MOST STABLE CLUSTER MATCHES:
 cluster_solution1  cluster_solution2  size_solution1  size_solution2  jaccard_similarity  overlap_pct
                 6                 17             572             568            0.935484    93.548387
                94                 69          

In [22]:
import pandas as pd

# Read the data
df = summary_df


# Create a cleaner version with better formatting
def format_solution_name(name):
    """Extract key parameters from solution name"""
    parts = name.replace("cluster_", "").split("_")
    alpha = parts[0].replace("alpha", "α=")
    k = parts[1].replace("k", "k=")
    res = parts[2].replace("res", "γ=")
    return f"{alpha}, {k}, {res}"


# Create formatted table
table_data = []
for idx, row in df.iterrows():
    table_data.append(
        {
            "Solution 1": format_solution_name(row["solution1"]),
            "Solution 2": format_solution_name(row["solution2"]),
            "Matched Clusters (n)": int(row["n_matched_clusters"]),
            "Mean Jaccard": f"{row['mean_jaccard']:.3f}",
            ">60% Overlap (%)": f"{row['pct_moderate_overlap_60']:.1f}",
            ">80% Overlap (%)": f"{row['pct_high_overlap_80']:.1f}",
            "ARI": f"{row['ARI']:.3f}",
            "NMI": f"{row['NMI']:.3f}",
        }
    )

table_df = pd.DataFrame(table_data)

# Save as CSV
table_df.to_excel(
    output_dir + "/cluster-qualifications/eTable_cluster_stability.xlsx", index=False
)

# Print for viewing
print(table_df.to_string(index=False))

# Also create a summary row
print("\n" + "=" * 80)
print("SUMMARY STATISTICS:")
print("=" * 80)
summary = {
    "Mean Jaccard": f"{df['mean_jaccard'].mean():.3f} ± {df['mean_jaccard'].std():.3f}",
    "Mean >60% Overlap": f"{df['pct_moderate_overlap_60'].mean():.1f}%",
    "Mean >80% Overlap": f"{df['pct_high_overlap_80'].mean():.1f}%",
    "Mean ARI": f"{df['ARI'].mean():.3f}",
    "Mean NMI": f"{df['NMI'].mean():.3f}",
}
for key, value in summary.items():
    print(f"{key:20s}: {value}")

          Solution 1           Solution 2  Matched Clusters (n) Mean Jaccard >60% Overlap (%) >80% Overlap (%)   ARI   NMI
α=0.3, k=20, γ=0.006 α=0.5, k=10, γ=0.002                    94        0.574             44.7             16.0 0.545 0.789
α=0.3, k=20, γ=0.006  α=0.3, k=5, γ=0.001                    86        0.516             37.2              8.1 0.464 0.750
α=0.3, k=20, γ=0.006 α=0.3, k=20, γ=0.005                   112        0.833             90.2             71.4 0.848 0.931
α=0.3, k=20, γ=0.006 α=0.3, k=10, γ=0.002                   100        0.675             71.0             37.0 0.673 0.846
α=0.5, k=10, γ=0.002  α=0.3, k=5, γ=0.001                    86        0.658             72.1             31.4 0.750 0.843
α=0.5, k=10, γ=0.002 α=0.3, k=20, γ=0.005                    94        0.547             44.7             17.0 0.574 0.787
α=0.5, k=10, γ=0.002 α=0.3, k=10, γ=0.002                    94        0.658             64.9             40.4 0.709 0.849
 α=0.3, k=5, γ=0